# 🔬 RAG Avanzado: técnicas de retrieval sobre Wikipedia económica

En este notebook aplicamos técnicas avanzadas de retrieval sobre un corpus de artículos de Wikipedia sobre economía laboral.

**Técnicas que veremos:**
1. **MMR** — evitar chunks redundantes
2. **Query Expansion** — generar variantes de la pregunta
3. **Re-ranking** — reordenar resultados con un segundo modelo
4. **Metadata Filtering** — restringir la búsqueda a un artículo específico

En cada sección compararemos el retrieval básico vs la técnica avanzada para ver la diferencia concreta.

> ⚠️ **Antes de empezar:** necesitas una API key de OpenRouter (gratis en [openrouter.ai](https://openrouter.ai))

## 0. Instalación de dependencias

In [1]:
%pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-chroma \
    sentence-transformers \
    wikipedia \
    rank-bm25 \
    flashrank

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuración

In [1]:
import os
import shutil
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("🔑 OpenRouter API Key: ")

MODELOS_FALLBACK = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "google/gemma-4-31b-it:free",
    "openai/gpt-oss-20b:free",
]

print("✅ Configuración lista")

✅ Configuración lista


## 2. Cargar corpus de Wikipedia

Descargamos 12 artículos de Wikipedia en español sobre economía laboral. Este corpus tiene características ideales para mostrar técnicas avanzadas:
- Artículos relacionados con vocabulario solapado (redundancia)
- Sinónimos frecuentes: *desempleo*, *desocupación*, *cesantía*
- Suficientes chunks para que el re-ranking tenga algo que reordenar

In [8]:
import time
from langchain_community.document_loaders import WikipediaLoader

temas = [
    "Desempleo",
    "Mercado de trabajo",
    "Tasa de actividad",
    "Empleo informal",
    "Salario mínimo",
    "Inflación",
    "Fuerza de trabajo",
    "Subempleo",
    "Brecha salarial de género",
    "Productividad laboral",
]

docs = []
for tema in temas:
    for intento in range(5):  # hasta 5 reintentos
        try:
            loader = WikipediaLoader(query=tema, lang="es", load_max_docs=1, doc_content_chars_max=50000)
            resultado = loader.load()
            docs.extend(resultado)
            print(f"  ✅ {tema}: {len(resultado[0].page_content):,} caracteres")
            time.sleep(3)  # pausa entre artículos para no saturar la API
            break
        except Exception as e:
            if intento < 4:
                print(f"  ⚠️  {tema}: reintentando ({intento+1}/5)...")
                time.sleep(5 * (intento + 1))  
                print(f"  ❌ {tema}: no se pudo cargar ({e})")

print(f"\n📄 Total artículos: {len(docs)}")
print(f"📝 Total caracteres: {sum(len(d.page_content) for d in docs):,}")


  ⚠️  Desempleo: reintentando (1/5)...
  ⚠️  Desempleo: reintentando (2/5)...
  ⚠️  Desempleo: reintentando (3/5)...
  ⚠️  Desempleo: reintentando (4/5)...
  ✅ Desempleo: 36,122 caracteres
  ✅ Mercado de trabajo: 7,105 caracteres
  ⚠️  Tasa de actividad: reintentando (1/5)...
  ⚠️  Tasa de actividad: reintentando (2/5)...
  ⚠️  Tasa de actividad: reintentando (3/5)...
  ✅ Tasa de actividad: 3,446 caracteres
  ✅ Empleo informal: 11,472 caracteres
  ✅ Salario mínimo: 50,000 caracteres
  ⚠️  Inflación: reintentando (1/5)...
  ⚠️  Inflación: reintentando (2/5)...
  ⚠️  Inflación: reintentando (3/5)...
  ⚠️  Inflación: reintentando (4/5)...
  ✅ Inflación: 46,973 caracteres
  ✅ Fuerza de trabajo: 7,365 caracteres
  ⚠️  Subempleo: reintentando (1/5)...
  ⚠️  Subempleo: reintentando (2/5)...
  ⚠️  Subempleo: reintentando (3/5)...
  ✅ Subempleo: 4,244 caracteres
  ✅ Brecha salarial de género: 31,521 caracteres
  ✅ Productividad laboral: 28,845 caracteres

📄 Total artículos: 10
📝 Total caractere

In [9]:
# Revisar metadata disponible — clave para el metadata filtering
print("Metadata de cada artículo:")
for doc in docs:
    titulo = doc.metadata.get('title', 'sin título')
    print(f"  - {titulo}: {len(doc.page_content):,} chars")

Metadata de cada artículo:
  - Desempleo: 36,122 chars
  - Mercado de trabajo: 7,105 chars
  - Población activa: 3,446 chars
  - Trabajo irregular: 11,472 chars
  - Salario mínimo: 50,000 chars
  - Inflación: 46,973 chars
  - Fuerza de trabajo: 7,365 chars
  - Subempleo: 4,244 chars
  - Brecha salarial de género: 31,521 chars
  - Productividad: 28,845 chars


## 3. Chunking e indexación base

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

if os.path.exists("./chroma_wiki"):
    shutil.rmtree("./chroma_wiki")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(docs)
print(f"✂️  Total chunks: {len(chunks)}")

print("\n⏳ Creando embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_wiki",
)
print(f"✅ ChromaDB listo con {vectorstore._collection.count()} vectores")

✂️  Total chunks: 726

⏳ Creando embeddings...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5457.01it/s]


✅ ChromaDB listo con 726 vectores


## 4. Helpers: LLM y función de consulta

In [39]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

modelo_activo = MODELOS_FALLBACK[2]

def crear_llm(modelo=None):
    return ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model=modelo or modelo_activo,
        temperature=0.0,
    )

prompt = PromptTemplate.from_template("""
Eres un asistente experto en economía laboral.
Responde la pregunta usando el siguiente contexto de Wikipedia.
Si el contexto tiene información parcial o relacionada, úsala para dar la mejor respuesta posible.
Solo responde "No encontré esa información" si el contexto no tiene absolutamente nada relevante.

Contexto:
{context}

Pregunta: {question}
Respuesta:
""")

def consultar(retriever_usado, modelo, pregunta):
    chain = (
            {"question": RunnablePassthrough(), "context": retriever_usado}
            | prompt
            | crear_llm(modelo)
        )
    respuesta = chain.invoke(pregunta)
    print(f"\n🤖 Respuesta:\n{respuesta.content}")


---
## Técnica 1: MMR — Maximal Marginal Relevance

**El problema:** similarity search básico puede devolver 5 chunks casi idénticos del mismo párrafo.

**La solución:** MMR balancea relevancia con diversidad — busca chunks relevantes que además sean distintos entre sí.

In [23]:
pregunta_mmr = "¿Qué es la inflación y cómo afecta al mercado laboral?"

# Retriever básico
retriever_basico = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# Retriever con MMR
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,       # candidatos iniciales
        "lambda_mult": 0.7   # 0=diversidad pura, 1=relevancia pura
    },
)

In [24]:
chunks_basico = retriever_basico.invoke(pregunta_mmr)
for c in chunks_basico:
    print(f"Chunk: \n{c.page_content[:200]}")

Chunk: 
== Efectos de la inflación ==
Chunk: 
para afectar los niveles salariales y que los empleados tienen capacidades similares.
Chunk: 
En economía, la inflación es el aumento sostenido y generalizado de los precios de los bienes y servicios en un mercado durante un período de tiempo, expresado en una determinada unidad monetaria.​ Cu
Chunk: 
== Jornada laboral y desempleo ==


== Desempleo en el mundo ==


=== Europa ===
Chunk: 
que si los salarios son rápidamente ajustados a la inflación se mitiga o elimina la pérdida de poder adquisitivo de algunos grupos sociales.


In [25]:
chunks_mmr = retriever_mmr.invoke(pregunta_mmr)
for c in chunks_mmr:
    print(f"Chunk: \n{c.page_content[:200]}")

Chunk: 
== Efectos de la inflación ==
Chunk: 
para afectar los niveles salariales y que los empleados tienen capacidades similares.
Chunk: 
En economía, la inflación es el aumento sostenido y generalizado de los precios de los bienes y servicios en un mercado durante un período de tiempo, expresado en una determinada unidad monetaria.​ Cu
Chunk: 
La oferta crece con el salario monetario a un ritmo decreciente

El nivel de precios influye en la oferta de trabajo sustituyendo el salario monetario W por su valor Pw

  
    
      
        
      
Chunk: 
Se denomina mercado de trabajo o mercado laboral al conjunto de relaciones entre empleadores (demandante es quien ofrece empleo) y personas que buscan trabajo remunerado oferentes. El mercado de traba


In [26]:
consultar(retriever_basico, modelo_activo, pregunta_mmr)


🤖 Respuesta:
**Inflación**  
En economía, la inflación es el aumento sostenido y generalizado de los precios de los bienes y servicios en un mercado durante un período de tiempo, expresado en una determinada unidad monetaria. Cuando el nivel general de precios se incrementa, el poder adquisitivo de la moneda disminuye, es decir, con la misma cantidad de dinero se pueden adquirir menos bienes y servicios.  

**Cómo afecta al mercado laboral**  

1. **Pérdida de poder adquisitivo** – Los salarios reales (ajustados por la inflación) pueden disminuir si los aumentos salariales no siguen el ritmo de la subida de precios. Esto reduce el poder de compra de los trabajadores y puede generar presión social y política para exigir ajustes salariales.

2. **Ajustes salariales y negociación colectiva** – En entornos de inflación moderada y estable, los contratos salariales suelen incluir cláusulas de ajuste automático (por ejemplo, índices de precios al consumidor) para proteger el poder adquisitiv

In [27]:
consultar(retriever_mmr, modelo_activo, pregunta_mmr)


🤖 Respuesta:
**Inflación**  
En economía, la inflación es el aumento sostenido y generalizado de los precios de los bienes y servicios en un mercado durante un período de tiempo, expresado en una determinada unidad monetaria. Cuando el nivel general de precios se incrementa, el poder adquisitivo de la moneda disminuye, ya que con la misma cantidad de dinero pueden adquirirse menos bienes y servicios.  

**Cómo afecta al mercado laboral**  

1. **Poder adquisitivo de los salarios**  
   - La inflación reduce el valor real de los salarios. Si los salarios nominales no crecen al mismo ritmo que la inflación, los trabajadores pierden poder adquisitivo, lo que puede generar insatisfacción y presión por aumentos salariales.

2. **Ajustes de salarios y negociación colectiva**  
   - En un entorno inflacionario, los sindicatos y los empleadores suelen negociar aumentos salariales que intenten compensar la pérdida de poder adquisitivo. Esto puede llevar a un ciclo de aumentos salariales y prec

---
## Técnica 2: Query Expansion

**El problema:** si el usuario escribe *"cesantía"* y el artículo usa *"desempleo"*, el retriever vectorial puede fallar porque las palabras tienen embeddings distintos.

**La solución:** el LLM genera variantes de la pregunta → se hacen múltiples búsquedas → se fusionan los resultados.

In [41]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Prompt para generar variantes de la pregunta
prompt_expansion = PromptTemplate.from_template("""
Genera 3 variantes de la siguiente pregunta usando sinónimos y reformulaciones distintas.
Por ejemplo: si la pregunta usa "cesantía" podrías usar "desempleo".
Devuelve SOLO las 3 variantes separadas por salto de línea, sin numeración ni explicación.

Pregunta original: {question}
""")

chain_expansion = prompt_expansion | crear_llm() | StrOutputParser()

def query_expansion(pregunta: str) -> list[str]:
    """Genera variantes de la pregunta usando el LLM."""
    resultado = chain_expansion.invoke({"question": pregunta})
    variantes = [v.strip() for v in resultado.strip().split("\n") if v.strip()]
    print("🔄 Variantes generadas:")
    for v in [pregunta] + variantes[:3]:
        print(f"  → {v}")
    return [pregunta] + variantes[:3]

def get_context_with_expansion(pregunta: str, k: int = 4) -> str:
    """Recupera chunks usando query expansion y los devuelve como string."""
    variantes = query_expansion(pregunta)
    retriever_base = vectorstore.as_retriever(search_kwargs={"k": k})
    
    chunks_vistos = {}  # page_content → chunk para deduplicar
    for variante in variantes:
        for chunk in retriever_base.invoke(variante):
            if chunk.page_content not in chunks_vistos:
                chunks_vistos[chunk.page_content] = chunk

    chunks_finales = list(chunks_vistos.values())[:k]
    
    print(f"\n📎 Chunks únicos recuperados ({len(chunks_finales)}):")
    for i, c in enumerate(chunks_finales, 1):
        titulo = c.metadata.get("title", c.metadata.get("source", "?"))
        print(f"  [{i}] {titulo}: {c.page_content[:120]}...")
    
    return "\n\n".join(c.page_content for c in chunks_finales)

# Chain con query expansion
chain_expansion_rag = (
    {
        "question": RunnablePassthrough(),
        "context": RunnableLambda(get_context_with_expansion),
    }
    | prompt
    | crear_llm()
    | StrOutputParser()
)

# Probar
pregunta = "¿Cómo afecta la automatización a los trabajadores?"
print(f"❓ Pregunta: {pregunta}\n")
respuesta = chain_expansion_rag.invoke(pregunta)
print(f"\n🤖 Respuesta:\n{respuesta}")

❓ Pregunta: ¿Cómo afecta la automatización a los trabajadores?

🔄 Variantes generadas:
  → ¿Cómo afecta la automatización a los trabajadores?
  → ¿Cómo influye la automatización en los empleados?
  → ¿Qué impacto tiene la automatización sobre la fuerza laboral?
  → De qué manera la automatización afecta a los profesionales?

📎 Chunks únicos recuperados (4):
  [1] Desempleo: para afectar los niveles salariales y que los empleados tienen capacidades similares....
  [2] Productividad: Dependencia tecnológica: la conexión permanente de las personas trabajadoras a un dispositivo electrónico (por ejemplo u...
  [3] Productividad: Factores humanos que afectan la productividad Su gente mantiene su negocio en funcionamiento, por lo que su bienestar y ...
  [4] Productividad: A menos que sus empleados realicen trabajos rutinarios de nivel inicial, su trabajo y productividad mejorarán con el tie...

🤖 Respuesta:
La automatización tiende a reducir la demanda de trabajos rutinarios de nivel inicial

---
## Técnica 3: Re-ranking con CrossEncoder

**El problema:** similitud coseno mide cercanía vectorial, pero no siempre refleja relevancia real para la pregunta específica.

**La solución:** un segundo modelo (CrossEncoder) relee cada chunk junto a la pregunta y los reordena por relevancia real. 

In [42]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('BAAI/bge-reranker-large', trust_remote_code=True)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2975.53it/s]


In [43]:
def retrieval_with_reranking(
    retriever,
    cross_encoder,
    top_k_retrieve=5,  # Cuántos recuperar del retriever
    top_k_rerank=3      # Cuántos devolver después del rerank
):
    
    def retrieve_and_rerank(query: str):
        # PASO 1: Recuperar candidatos del retriever
        unique_docs = retriever.invoke(query)
        
        # Limitar a top_k_retrieve
        unique_docs = unique_docs[:top_k_retrieve]
        
        # PASO 3: Neural reranking con cross-encoder
        texts = [doc.page_content for doc in unique_docs]
        sentence_pairs = [[query, text] for text in texts]
        
        # Calcular scores de relevancia
        scores = cross_encoder.predict(sentence_pairs)
        
        # PASO 4: Ordenar por score y devolver top_k_rerank
        ranked = sorted(
            zip(scores, unique_docs),
            key=lambda x: x[0],
            reverse=True
        )
        
        return [doc for _, doc in ranked[:top_k_rerank]]
    
    return RunnableLambda(retrieve_and_rerank)

In [47]:
retriever_rerank = retrieval_with_reranking(
    retriever=retriever_mmr,
    cross_encoder=cross_encoder,
    top_k_retrieve=10,  
    top_k_rerank=5
    )

In [45]:
pregunta_rerank = "¿Cómo afecta el salario mínimo al empleo informal?"

resultado_mmr = retriever_mmr.invoke(pregunta_rerank)
for c in resultado_mmr:
    print(c.page_content)


En el país se han intentado llevar a cabo leyes como la regulación de empresas, creación de censos de informalidad para el incentivo a la seguridad social, reformas tributarias más laxas, incluso exenciones, pero no han tenido un efecto relevante ante el fenómeno del empleo informal. Hay algunas cuestiones que pueden verse como regulaciones pero a la vez son barreras, como lo es el salario mínimo. Actualmente el salario mínimo ha sido un gran debate en cuestión de la necesidad de ingresos y lo
== Empleo informal en México ==
El empleo informal es todo empleo que no está registrado, regulado o protegido por marcos legales o normativos, ya sea un trabajo remunerado o no remunerado.​​ Es decir, todo aquel empleo no registrado o bien inexactamente registrado (falsificación en fecha de ingreso en los recibos de sueldo, falsa remuneración) y  que por tanto no se encuentra en regla con las distintas normas o legislaciones. También se puede referir a ello coloquialmente como  trabajo en negro,

In [48]:
restultado_rerank = retriever_rerank.invoke(pregunta_rerank)
for c in restultado_rerank:
    print(c.page_content)

El salario es el producto marginal decreciente del trabajo
El salario mínimo afecta también a la calidad del empleo, dado su impacto en el tipo de puestos de trabajo ofrecidos. Desde este punto de vista, la fijación de un salario mínimo puede mejorar la asignación de recursos de la economía, estimulando la creación de empleos con mayores niveles de productividad.
En el país se han intentado llevar a cabo leyes como la regulación de empresas, creación de censos de informalidad para el incentivo a la seguridad social, reformas tributarias más laxas, incluso exenciones, pero no han tenido un efecto relevante ante el fenómeno del empleo informal. Hay algunas cuestiones que pueden verse como regulaciones pero a la vez son barreras, como lo es el salario mínimo. Actualmente el salario mínimo ha sido un gran debate en cuestión de la necesidad de ingresos y lo
El empleo informal es todo empleo que no está registrado, regulado o protegido por marcos legales o normativos, ya sea un trabajo remun

In [50]:
consultar(retriever_mmr, modelo_activo, pregunta_rerank)


🤖 Respuesta:
El salario mínimo, al establecer un piso legal sobre la remuneración que los empleadores deben pagar, tiene efectos directos e indirectos sobre el empleo informal:

1. **Incentivo a la formalización**  
   - Cuando el salario mínimo se sitúa en un nivel que los empleadores pueden cubrir sin incurrir en costos adicionales significativos, la diferencia entre trabajar formalmente y en negro se reduce.  
   - Los trabajadores que antes aceptaban salarios por debajo del mínimo pueden exigir la regularización para recibir la remuneración mínima obligatoria, lo que aumenta la formalidad del empleo.

2. **Presión sobre la informalidad cuando el mínimo es elevado**  
   - Si el salario mínimo se fija por encima del nivel que el mercado informal puede sostener (por ejemplo, en sectores con alta elasticidad de la demanda laboral), los empleadores informales pueden verse forzados a reducir la contratación o a trasladar la actividad a la informalidad para evitar el pago del mínimo.  


In [49]:
consultar(retriever_rerank, modelo_activo, pregunta_rerank)


🤖 Respuesta:
El salario mínimo influye en el empleo informal de varias maneras:

1. **Presión sobre la informalidad**  
   - Cuando el salario mínimo se fija por encima del nivel de remuneración que los empleadores informales están dispuestos o pueden pagar, aumenta la brecha entre el trabajo formal y el informal.  
   - Los trabajadores que no pueden acceder a empleos formales con salarios que cumplan con el mínimo pueden verse forzados a permanecer en la informalidad, donde no están protegidos por la ley y no reciben el salario mínimo.

2. **Incentivo a la formalización**  
   - En algunos casos, un salario mínimo más alto puede motivar a los empleadores informales a formalizar sus negocios para poder cumplir con la normativa y evitar sanciones.  
   - Sin embargo, el aumento del costo laboral también puede hacer que la formalización sea económicamente inviable para pequeñas empresas, perpetuando la informalidad.

3. **Impacto en la calidad del empleo**  
   - La fijación de un sala

---
## Técnica 4: Metadata Filtering

**El problema:** a veces sabes exactamente en qué artículo está la respuesta y no quieres que el retriever busque en todo el corpus.

**La solución:** filtrar por metadata antes de buscar — más rápido y más preciso cuando el origen importa.

In [51]:
# Ver los títulos exactos disponibles para filtrar
titulos = list(set(c.metadata.get('title', '') for c in chunks))
print("Títulos disponibles para filtrar:")
for t in sorted(titulos):
    print(f"  - {t}")

Títulos disponibles para filtrar:
  - Brecha salarial de género
  - Desempleo
  - Fuerza de trabajo
  - Inflación
  - Mercado de trabajo
  - Población activa
  - Productividad
  - Salario mínimo
  - Subempleo
  - Trabajo irregular


In [75]:
pregunta_filtro = "¿Cómo se calcula y para qué sirve el IPC?"

# Sin filtro: busca en todo el corpus
retriever_sin_filtro = vectorstore.as_retriever(search_kwargs={"k": 4})

# Con filtro: solo en el artículo de brecha salarial
# ⚠️ El título debe coincidir exactamente con lo que devolvió la celda anterior
retriever_filtrado = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "filter": {"title": "Inflación"}
    }
)


In [76]:
respuesta_sin_filtro = retriever_sin_filtro.invoke(pregunta_filtro)
for c in respuesta_sin_filtro:
    print(c.page_content)

10 %, entre 4 y 6 meses si el IPC era de entre 10 % y 23 %, y entre 3 y 4 meses si el IPC era mayor a 23 % anual.
La inflación se calcula generalmente mediante la tasa de variación del índice de precios en el tiempo, por lo general el Índice de Precios al Consumidor, que mide los precios de una selección de bienes y servicios adquiridos por un consumidor medio.
Por ejemplo, en enero de 2007, el Índice de Precios al Consumidor de los EE. UU. fue 202,416, y en enero de 2008 era 211,080. La fórmula para calcular el porcentaje de la tasa de inflación anual del IPC a lo largo de 2007 es entonces
Índice de Precios al Consumidor (IPC): Es un indicador social y económico de coyuntura, construido para medir los cambios experimentados a lo largo del tiempo en relación con el nivel general de precios de los bienes y servicios de consumo que los hogares pagan, adquieren o utilizan para ser consumidos. La Resolución sobre índices de los precios al consumidor adoptada por la decimoséptima Conferenci

In [77]:
respuesta_con_filtro = retriever_filtrado.invoke(pregunta_filtro)
for c in respuesta_con_filtro:
    print(c.page_content)

La inflación se calcula generalmente mediante la tasa de variación del índice de precios en el tiempo, por lo general el Índice de Precios al Consumidor, que mide los precios de una selección de bienes y servicios adquiridos por un consumidor medio.
Por ejemplo, en enero de 2007, el Índice de Precios al Consumidor de los EE. UU. fue 202,416, y en enero de 2008 era 211,080. La fórmula para calcular el porcentaje de la tasa de inflación anual del IPC a lo largo de 2007 es entonces
La tasa de inflación resultante del IPC en el período de un año es de 4,28 %. Es decir, el nivel general de precios a los consumidores aumentó aproximadamente cuatro por ciento en 2007.
Para obtener la inflación de un año determinado se toma como base el índice de precios de diciembre del año anterior y se lo compara con el de diciembre del último año.


== Causas de la inflación ==


=== Teoría monetaria ===
== Mediciones ==
== Efectos de la inflación ==


In [78]:
consultar(retriever_sin_filtro, modelo_activo, pregunta_filtro)


🤖 Respuesta:
El **Índice de Precios al Consumidor (IPC)** se calcula como la variación porcentual del precio de una canasta de bienes y servicios que representa el consumo típico de los hogares.  
Para su cálculo se:

1. **Selecciona una canasta de referencia** con los productos y servicios que los consumidores adquieren habitualmente (alimentos, vivienda, transporte, salud, educación, etc.).  
2. **Registra los precios** de cada artículo de la canasta en un periodo base y en el periodo actual.  
3. **Pondera** cada precio según la importancia relativa de ese artículo en el gasto total de los hogares.  
4. **Calcula el índice** comparando el valor total de la canasta en el periodo actual con el valor de la canasta en el periodo base, y expresa la diferencia como un porcentaje.

---

### ¿Para qué sirve el IPC?

1. **Medir la inflación**: El IPC es el indicador más utilizado para cuantificar la inflación, es decir, el aumento sostenido y generalizado de los precios.  
2. **Ajustar sala

In [79]:
consultar(retriever_filtrado, modelo_activo, pregunta_filtro)


🤖 Respuesta:
El **Índice de Precios al Consumidor (IPC)** es la medida más utilizada para cuantificar la inflación en un país.  

### 1. ¿Cómo se calcula?

1. **Selección de la canasta de bienes y servicios**  
   - Se elige una canasta representativa de los productos y servicios que consume un hogar típico (alimentos, vivienda, transporte, salud, educación, ocio, etc.).  
   - Cada elemento de la canasta recibe un **peso** que refleja su importancia relativa en el gasto medio del consumidor.

2. **Obtención de precios**  
   - Se recogen los precios de cada artículo de la canasta en un periodo de referencia (por ejemplo, cada mes).  
   - Se comparan con los precios del mismo periodo del año anterior (o de un mes base).

3. **Cálculo del índice**  
   - Se calcula el **precio ponderado** de la canasta para el periodo actual y para el periodo base.  
   - El IPC se expresa como un índice numérico (por ejemplo, 100 en el año base).  
   - La fórmula típica es:  

     \[
     \text{IPC